In [0]:
%pip install soccerdata --upgrade

In [0]:
%restart_python

In [0]:
from pyspark.sql.functions import current_timestamp, lit
import requests

# 1. Configuração dos cabeçalhos com o domínio correto (www.sofascore.com)
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.sofascore.com/",
    "Accept": "application/json, text/plain, */*",
}

event_id = 15235478
url = f"https://www.sofascore.com/api/v1/event/{event_id}/lineups"

response = requests.get(url, headers=headers)

if response.status_code == 200:
  dados = response.json()
  jogadores_extraidos = []

  # Processa os times mandante (home) e visitante (away)
  for lado in ["home", "away"]:
    lista_jogadores = dados.get(lado, {}).get("players", [])

    for item in lista_jogadores:
      player = item.get("player", {})
      stats = item.get("statistics", {})

      # Captura tanto titulares quanto reservas que entraram e tiveram estatísticas
      jogadores_extraidos.append({
          "event_id": event_id,
          "side": lado,
          "player_id": player.get("id"),
          "player_name": player.get("name"),
          "slug": player.get("slug"),
          "position": item.get("position"),
          "jersey_number": item.get("jerseyNumber"),
          "substitute": item.get("substitute", False),
          "captain": item.get("captain", False),
          "rating": stats.get("rating"),
          "minutes_played": stats.get("minutesPlayed"),
          "goals": stats.get("goals", 0),
          "assists": stats.get("goalAssist", 0),
          "expected_goals": stats.get("expectedGoals"),
          "expected_assists": stats.get("expectedAssists"),
          "key_passes": stats.get("keyPass", 0),
          "total_passes": stats.get("totalPass"),
          "accurate_passes": stats.get("accuratePass"),
          "touches": stats.get("touches"),
      })

  # 2. Converte para Spark DataFrame e adiciona colunas de auditoria (Bronze)
  df_spark = (
      spark.createDataFrame(jogadores_extraidos)
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_url", lit(url))
  )

  # 3. Persiste como tabela Delta no catálogo
  spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
  df_spark.write.format("delta").mode("overwrite").saveAsTable(
      "bronze.sofascore_event_lineups"
  )

  print(
      f"[SUCESSO] Tabela 'bronze.sofascore_event_lineups' criada com"
      f" {df_spark.count()} atletas."
  )
  display(
      df_spark.select(
          "player_name",
          "side",
          "position",
          "rating",
          "expected_goals",
          "minutes_played",
      )
  )

else:
  print(f"[ERRO] Requisição falhou com status {response.status_code}:")
  print(response.text[:300])

In [0]:
%sh
# Testa resolução de DNS e conectividade básica
curl -I -m 5 https://www.google.com
curl -I -m 5 https://www.sofascore.com